In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
# ─── CONFIG ──────────────────────────────────────────────────────────────
catalog             = "charles_schwab_retailbrokerage_dev_team_lemma"
bronze_customermgmt = f"{catalog}.bronze.customermgmt"
bronze_customer_txt = f"{catalog}.bronze.customer"
staging_scd2        = f"{catalog}.staging.customer_scd2_versions"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.staging")

# Widget for batch tracking
dbutils.widgets.text("batch_id", "1", "Batch ID")
current_batch = dbutils.widgets.get("batch_id")

# try:
#     run_id_row = spark.sql(f"SELECT _run_id FROM {bronze_customermgmt} LIMIT 1").first()
#     carried_run_id = run_id_row[0] if run_id_row else "unknown"
# except:
#     carried_run_id = "unknown"

try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch_id FROM {bronze_customermgmt} LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "unknown"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "unknown"

spark.conf.set("spark.sql.adaptive.enabled", "true")

In [0]:
# ─── STEP 1: READ BRONZE & UNION (ALL HISTORICAL EVENTS) ─────────────────
df_combined = spark.sql(f"""
    WITH b1_xml AS (
        SELECT 
            CAST(ActionTS AS TIMESTAMP) as action_ts,
            CAST(C_ID AS BIGINT) as customerid,
            C_TAX_ID as taxid, C_GNDR as gender, TRY_CAST(C_TIER AS TINYINT) as tier, CAST(C_DOB AS DATE) as dob,
            C_L_NAME as lastname, C_F_NAME as firstname, C_M_NAME as middleinitial, 
            C_ADLINE1 as addressline1, C_ADLINE2 as addressline2, C_ZIPCODE as postalcode, 
            C_CITY as city, C_STATE_PROV as stateprov, C_CTRY as country,
            C_PRIM_EMAIL as primaryemail, C_ALT_EMAIL as alternateemail,
            C_CTRY_1, C_AREA_1, C_LOCAL_1, C_EXT_1,
            C_CTRY_2, C_AREA_2, C_LOCAL_2, C_EXT_2,
            C_CTRY_3, C_AREA_3, C_LOCAL_3, C_EXT_3,
            C_LCL_TX_ID, C_NAT_TX_ID,
            _batch_id as _batch, _run_id
        FROM {bronze_customermgmt} 
        WHERE ActionType IN ('NEW', 'UPDCUST', 'INACT')
    ),
    b2_txt AS (
        SELECT 
            CAST(_ingest_ts AS TIMESTAMP) as action_ts,
            CAST(C_ID AS BIGINT) as customerid,
            C_TAX_ID as taxid, C_GNDR as gender, TRY_CAST(C_TIER AS TINYINT) as tier, CAST(C_DOB AS DATE) as dob,
            C_L_NAME as lastname, C_F_NAME as firstname, C_M_NAME as middleinitial, 
            C_ADLINE1 as addressline1, C_ADLINE2 as addressline2, C_ZIPCODE as postalcode, 
            C_CITY as city, C_STATE_PROV as stateprov, C_CTRY as country,
            C_EMAIL_1 as primaryemail, C_EMAIL_2 as alternateemail,
            C_CTRY_1, C_AREA_1, C_LOCAL_1, C_EXT_1,
            C_CTRY_2, C_AREA_2, C_LOCAL_2, C_EXT_2,
            C_CTRY_3, C_AREA_3, C_LOCAL_3, C_EXT_3,
            C_LCL_TX_ID, C_NAT_TX_ID,
            _batch_id as _batch, _run_id
        FROM {bronze_customer_txt}
    )
    SELECT * FROM b1_xml UNION ALL SELECT * FROM b2_txt
""")
df_combined.createOrReplaceTempView("v_combined_raw")

In [0]:
# ─── STEP 2: FORWARD-FILL NULLS & COMPUTE SCD-2 WINDOWS ──────────────────
# TPC-DI requires forward-filling missing attributes in CDC updates.
df_staging = spark.sql(f"""
    WITH forward_filled AS (
        SELECT 
            action_ts, customerid,
            LAST_VALUE(taxid, true) OVER w AS taxid,
            LAST_VALUE(gender, true) OVER w AS gender,
            LAST_VALUE(tier, true) OVER w AS tier,
            LAST_VALUE(dob, true) OVER w AS dob,
            LAST_VALUE(lastname, true) OVER w AS lastname,
            LAST_VALUE(firstname, true) OVER w AS firstname,
            LAST_VALUE(middleinitial, true) OVER w AS middleinitial,
            LAST_VALUE(addressline1, true) OVER w AS addressline1,
            LAST_VALUE(addressline2, true) OVER w AS addressline2,
            LAST_VALUE(postalcode, true) OVER w AS postalcode,
            LAST_VALUE(city, true) OVER w AS city,
            LAST_VALUE(stateprov, true) OVER w AS stateprov,
            LAST_VALUE(country, true) OVER w AS country,
            LAST_VALUE(primaryemail, true) OVER w AS primaryemail,
            LAST_VALUE(alternateemail, true) OVER w AS alternateemail,
            LAST_VALUE(C_CTRY_1, true) OVER w AS c_ctry_1, LAST_VALUE(C_AREA_1, true) OVER w AS c_area_1, LAST_VALUE(C_LOCAL_1, true) OVER w AS c_local_1,
            LAST_VALUE(C_CTRY_2, true) OVER w AS c_ctry_2, LAST_VALUE(C_AREA_2, true) OVER w AS c_area_2, LAST_VALUE(C_LOCAL_2, true) OVER w AS c_local_2,
            LAST_VALUE(C_CTRY_3, true) OVER w AS c_ctry_3, LAST_VALUE(C_AREA_3, true) OVER w AS c_area_3, LAST_VALUE(C_LOCAL_3, true) OVER w AS c_local_3,
            LAST_VALUE(C_LCL_TX_ID, true) OVER w AS c_lcl_tx_id,
            LAST_VALUE(C_NAT_TX_ID, true) OVER w AS c_nat_tx_id,
            _batch, _run_id
        FROM v_combined_raw
        WINDOW w AS (PARTITION BY customerid ORDER BY action_ts ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
    )
    SELECT 
        *,
        CAST(action_ts AS DATE) AS effectivedate,
        COALESCE(
            CAST(LEAD(action_ts) OVER (PARTITION BY customerid ORDER BY action_ts) AS DATE),
            CAST('9999-12-31' AS DATE)
        ) AS enddate,
        ROW_NUMBER() OVER (PARTITION BY customerid ORDER BY action_ts) AS version_number,
        CASE WHEN LEAD(action_ts) OVER (PARTITION BY customerid ORDER BY action_ts) IS NULL THEN TRUE ELSE FALSE END AS iscurrent,
        SHA2(CONCAT_WS('||', customerid, lastname, addressline1, action_ts), 256) AS record_hash,
        CURRENT_TIMESTAMP() AS _load_ts
    FROM forward_filled
""")

df_staging.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(staging_scd2)

In [0]:
# ─── STEP 3: Execute Operations Logging ────────────────────────────────────

# 1. Get the row counts for reconciliation
raw_count = spark.sql("SELECT COUNT(*) FROM v_combined_raw").first()[0]
staging_count = spark.sql(f"SELECT COUNT(*) FROM {staging_scd2}").first()[0]

# 2. Log the Audit Event (Tracks the action taken)
log_audit_event(
    spark=spark, 
    run_id=carried_run_id, 
    batch=current_batch, 
    layer="staging", 
    table_name="customer_scd2_versions", 
    operation="OVERWRITE", 
    rows_affected=staging_count
)

# 3. Log the Pipeline Reconciliation (Tracks data loss/inflation between layers)
log_pipeline_recon(
    spark=spark, 
    run_id=carried_run_id, 
    batch_id=current_batch, 
    domain="CUSTOMER", 
    table_name="customer_scd2_versions", 
    source_layer="bronze", 
    target_layer="staging", 
    source_count=raw_count, 
    target_count=staging_count
)

log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'staging_customer_customermgmt', 'Successfully completed processing for customer SCD-2 workspace.')

